In [2]:
import jax
import jax.numpy as jnp
import netket as nk
import flax
from transformers import FlaxAutoModel

#required to prevent checkpointing conflicts with HuggingFace (?)
flax.config.update('flax_use_orbax_checkpointing', False)

#download and load pre-trained model
print("Loading pre-trained model\n...")
p = 0.7 #* fix the value of the external field
L = 6
revision = f"L{L}_p{p}"
trial_model = FlaxAutoModel.from_pretrained("nqs-models/heisenberg_disorder_fnqs", trust_remote_code = True, revision=revision)
modify_model = FlaxAutoModel.from_pretrained("nqs-models/heisenberg_disorder_fnqs", trust_remote_code = True, revision=revision)

print("Define physical system (for testing)\n...")

#Standard FNQS models are often trained on 10x10 grids (100 spins), much greater than our samples??
#10x10 square lattice with periodic boundary conditions
graph = nk.graph.Square(10,pbc=True)
#print(f"\tHelp: nk.graph: {help(nk.graph)}\n")
hi = nk.hilbert.Spin(s=0.5, N=graph.n_nodes)

print("Define Hamiltonian\n...")

#create Heisenberg hamiltonian
H = nk.operator.Heisenberg(hilbert=hi, graph=graph)

print("Initialize Monte Carlo State\n...")

#set up Markov Chain Monte Carlo sampler that preserves total magnetization
sampler = nk.sampler.MetropolisExchange(hilbert=hi, graph=graph)

#wrap the model in a netket variational state
#critical: we inject the pre-trained weights using the 'variables' argument
vstate = nk.vqs.MCState(
    sampler = sampler,
    model=trial_model,
    n_samples=1024,
    variables={'params':trial_model.params}
)

print("Running Evaluation\n...")

energy = vstate.expect(H)

print(f"\nVerification Complete:\n\tEnergy Expectation Value: {energy}")


Loading pre-trained model
...


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Define physical system (for testing)
...
Define Hamiltonian
...
Initialize Monte Carlo State
...


TypeError: ViTFNQSModel.__call__() missing 1 required positional argument: 'coups'

In [2]:
help(nk.operator.Ising)

Help on class IsingJax in module netket.operator._ising.jax:

class IsingJax(netket.operator._ising.base.IsingBase, netket.operator._discrete_operator_jax.DiscreteJaxOperator)
 |  IsingJax(hilbert: netket.hilbert.abstract_hilbert.AbstractHilbert, graph: Union[netket.graph.abstract_graph.AbstractGraph, numpy.ndarray, jax.jaxlib._jax.Array], h: float, J: float, dtype: Union[NoneType, str, type[Any], numpy.dtype, netket.utils.types._SupportsDType])
 |  
 |  Jax-based implementation of the Transverse-Field Ising Hamiltonian
 |  :math:`-h\sum_i \sigma_i^{(x)} +J\sum_{\langle i,j\rangle} \sigma_i^{(z)}\sigma_j^{(z)}`.
 |  
 |  This implementation is considerably faster than the
 |  Ising hamiltonian constructed by summing
 |  :class:`~netket.operator.LocalOperator` s.
 |  
 |  Method resolution order:
 |      IsingJax
 |      netket.operator._ising.base.IsingBase
 |      netket.operator._hamiltonian.SpecialHamiltonian
 |      netket.operator._discrete_operator_jax.DiscreteJaxOperator
 |     

In [ ]:
from functools import partial
import jax
import jax.numpy as jnp
import netket as nk
import math
import flax
from flax.training import checkpoints
import numpy as np
from netket.operator.spin import sigmax, sigmaz, sigmay

flax.config.update('flax_use_orbax_checkpointing', False)

p = 0.7 #* fix the value of the external field
L = 6
revision = f"L{L}_p{p}"

def edges_square_lattice(L):
    Ns = L*L
    indices = np.arange(Ns)
    indices_right = (indices+1)%L + L*(indices//L)
    indices_down = (indices+L)%Ns
    first = np.c_[indices, indices_right]
    second = np.c_[indices, indices_down]

    edges = np.concatenate([first, second], axis=0)
    return edges

def coupling_heis_random(random_J, edges):
    edges_with_random_vars = list(zip(edges, random_J))
    return edges_with_random_vars

def si_sj(hi, i, j, txy=1.0):
    # 0.25 factor is to take into account for spin operators
    return 0.25*(txy * (sigmax(hi, i) * sigmax(hi, j) + sigmay(hi, i) * sigmay(hi, j)) + sigmaz(hi, i) * sigmaz(hi, j))

def heisenberg_hamiltonian(edges_Js, hi, txy=1.0):
    ham = 0.0
    for (ij, J) in edges_Js:
        ham += J * si_sj(hi,  ij[0], ij[1], txy)
    return ham

from transformers import FlaxAutoModel
wf = FlaxAutoModel.from_pretrained("nqs-models/heisenberg_disorder_fnqs", 
                                   trust_remote_code=True, 
                                   revision=revision)

N_params = nk.jax.tree_size(wf.params)
print('Number of parameters = ', N_params, flush=True)

lattice = nk.graph.Hypercube(length=L, n_dim=2, pbc=True)
hilbert = nk.hilbert.Spin(s=1/2, N=lattice.n_nodes, total_sz=0)

# Random Heisenberg Hamiltonian
from huggingface_hub import hf_hub_download
coups_path = hf_hub_download(repo_id="nqs-models/heisenberg_disorder_fnqs", filename="coups", revision=revision)
random_J = np.loadtxt(coups_path)[0]
edges = edges_square_lattice(L)
edges_Js = coupling_heis_random(random_J=random_J, edges=edges)

N_mc = 6000

hamiltonian = heisenberg_hamiltonian(edges_Js, hilbert)
sampler = nk.sampler.MetropolisExchange(hilbert=hilbert,
                                        graph=lattice,
                                        d_max=2,
                                        n_chains=N_mc,
                                        sweep_size=lattice.n_nodes)

key = jax.random.key(0)
key, subkey = jax.random.split(key, 2)
vstate = nk.vqs.MCState(sampler=sampler, 
                        apply_fun=partial(wf.__call__, coups=random_J), 
                        sampler_seed=subkey,
                        n_samples=N_mc, 
                        n_discard_per_chain=0,
                        variables=wf.params,
                        chunk_size=N_mc)

path = hf_hub_download(repo_id="nqs-models/heisenberg_disorder_fnqs", filename="spins", revision=revision)
samples = checkpoints.restore_checkpoint(path, target=None)
samples = jnp.array(samples, dtype='int8')
vstate.sampler_state = vstate.sampler_state.replace(σ = samples)

import time
# Sample the model
for _ in range(10):
    start = time.time()
    E = vstate.expect(hamiltonian)
    vstate.sample()

    print("Mean: ", E.mean.real / lattice.n_nodes, "\t time=", time.time()-start, flush=True)


C:\Users\chait\AppData\Local\hermes\hermes-agent\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chait\.cache\huggingface\hub\models--nqs-models--heisenberg_disorder_fnqs. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
The argument `trust_remote_code` is to be used with Auto classes. It has no eff

Number of parameters =  793240


C:\Users\chait\AppData\Local\hermes\hermes-agent\venv\Lib\site-packages\netket\operator\_local_operator\base.py:348: OperatorMultiplicationDeprecationWarning: 
The '*' operator for multiplying operators is deprecated and will be removed in a future version.

Please use the '@' operator instead:
  - Replace: operator1 * operator2
  - With:    operator1 @ operator2

The '@' operator is Python's standard matrix multiplication operator.


-------------------------------------------------------
For more detailed informations, visit the following link:
	 https://netket.readthedocs.io/en/latest/api/_generated/errors/netket.errors.OperatorMultiplicationDeprecationWarning.html
or the list of all common errors and warnings at
	 https://netket.readthedocs.io/en/latest/api/errors.html
-------------------------------------------------------

  warnings.warn(OperatorMultiplicationDeprecationWarning())
C:\Users\chait\AppData\Local\hermes\hermes-agent\venv\Lib\site-packages\netket\operator\_local_oper

Mean:  